In [9]:
import pandas as pd
import numpy as np
!pip install duckdb 
import duckdb

In [37]:
data = pd.read_csv(r"C:\Users\UDAY BISHT\Downloads\QVI_data.csv")

data['DATE'] = pd.to_datetime(data['DATE'])
data = data.dropna()

data = data[data['PROD_QTY'] < 100]

Q1 = data['TOT_SALES'].quantile(0.25)
Q3 = data['TOT_SALES'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

data = data[(data['TOT_SALES'] >= lower_bound) & (data['TOT_SALES'] <= upper_bound)]
data.drop_duplicates()
print(f"Data cleaned! New number of rows: {data.shape[0]}")
data.head()

Data cleaned! New number of rows: 264258


,LYLTY_CARD_NBR,DATE,STORE_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES,PACK_SIZE,BRAND,LIFESTAGE,PREMIUM_CUSTOMER
0,1000,2018-10-17,1,1,5,Natural Chip Compny SeaSalt175g,2,6.0,175,NATURAL,YOUNG SINGLES/COUPLES,Premium
1,1002,2018-09-16,1,2,58,Red Rock Deli Chikn&Garlic Aioli 150g,1,2.7,150,RRD,YOUNG SINGLES/COUPLES,Mainstream
2,1003,2019-03-07,1,3,52,Grain Waves Sour Cream&Chives 210G,1,3.6,210,GRNWVES,YOUNG FAMILIES,Budget
3,1003,2019-03-08,1,4,106,Natural ChipCo Hony Soy Chckn175g,1,3.0,175,NATURAL,YOUNG FAMILIES,Budget
4,1004,2018-11-02,1,5,96,WW Original Stacked Chips 160g,1,1.9,160,WOOLWORTHS,OLDER SINGLES/COUPLES,Mainstream


In [42]:
# here we are creating a function to check the sales of the trail stores before the trail started 

def pre_trail_sales(store_number):
    store_data = data [
        (data ['STORE_NBR'] == store_number) &
        (data['DATE'] <= '2019-01-30') &
        (data['DATE'] >= '2018-07-01')
    ]
    total_sales = store_data['TOT_SALES'].sum()
    return total_sales

store_to_check = [77,86,88]
for store in store_to_check:
    sales = pre_trail_sales(store)
    print (f'Store{store} Pre-Trail Sales:{sales.round(2)}')

Store77 Pre-Trail Sales:1693.6
Store86 Pre-Trail Sales:6091.45
Store88 Pre-Trail Sales:9293.6


In [55]:
# now we are finding the most similar stores to the trail stores 
trial_stores = [77, 86, 88]
all_stores = data['STORE_NBR'].unique()
for trial in trial_stores:
    trial_sales = pre_trail_sales(trial)
    min_diff = float('inf')
    best_control = 0
    
    for store in all_stores:
        if store in trial_stores:
            continue
            
        curr_sales = pre_trail_sales(store)
        diff = abs(trial_sales - curr_sales)
        
        if diff < min_diff:
            min_diff = diff
            best_control = store
            
    print(f"Trial Store {trial} -> Best Control Store: {best_control} (Diff: ${min_diff.round(2)})")

Trial Store 77 -> Best Control Store: 187 (Diff: $9.4)
Trial Store 86 -> Best Control Store: 196 (Diff: $3.45)
Trial Store 88 -> Best Control Store: 237 (Diff: $37.2)


In [63]:
# finding the scaling factor of the respactive stores
def scaling_factor(trial_store, control_store):
    trial_sales = pre_trail_sales(trial_store)
    control_sales = pre_trail_sales(control_store)
    
    if control_sales == 0:
        return 0.0
        
    return trial_sales / control_sales
print("\n--- Scaling Factors ---")
for trial, control in trail_control_dict.items():
    scale_fact = scaling_factor(trial, control)
    print(f'Trial Store {trial} & Control Store {control} -> Scaling Factor is: {scale_fact.round(4)}')


--- Scaling Factors ---
Trial Store 77 & Control Store 187 -> Scaling Factor is: 1.0056
Trial Store 86 & Control Store 196 -> Scaling Factor is: 0.9994
Trial Store 88 & Control Store 237 -> Scaling Factor is: 1.004


In [44]:
# finding the sales made in the trail period in both trail and control store 
def post_trail_sales(store_number):
    store_data = data[
        (data['STORE_NBR'] == store_number) &
        (data['DATE'] <= '2019-04-30')
        (data['DATE'] >= '2019-02-01')
    ]
    total_sales = store_data['TOT_SALES'].sum()
    return total_sales

stores_to_check = [77,86,88, 187,196,237]

for store in stores_to_check :
    curr_sales = post_trail_sales(store)
    print(f'The Store is: {store} And the total sales made in trail period is: {curr_sales.round(2)}') 

The Store is: 77 And the total sales made in trail period is: 777.0
The Store is: 86 And the total sales made in trail period is: 2788.2
The Store is: 88 And the total sales made in trail period is: 4286.8
The Store is: 187 And the total sales made in trail period is: 733.9
The Store is: 196 And the total sales made in trail period is: 2619.4
The Store is: 237 And the total sales made in trail period is: 3817.6


In [66]:
# showing the improvement in the trail stores compare to the control stores
print("--- Trial Store Performance vs. Scaled Control ---")

for trial, control in trail_control_dict.items():
    
    scale_fact = scaling_factor(trial, control)
    control_trial_sales = post_trail_sales(control)
    scaled_sales = control_trial_sales * scale_fact
    trial_actual_sales = post_trail_sales(trial)
    
    difference = trial_actual_sales - scaled_sales
    
    if scaled_sales > 0:
        improvement_pct = (difference / scaled_sales) * 100
    else:
        improvement_pct = 0.0
    print(f"Trial Store: {trial} | Control Store: {control}")
    print(f"  Scaled Control Sales (Expected): ${scaled_sales.round(2)}")
    print(f"  Trial Store Actual Sales:        ${trial_actual_sales.round(2)}")
    
    if difference > 0:
        print(f"  Sales Difference:                +${difference.round(2)}")
        print(f"  Performance Improvement:         +{improvement_pct.round(2)}%\n")
    else:
        print(f"  Sales Difference:                ${difference.round(2)}")
        print(f"  Performance Improvement:         {improvement_pct.round(2)}%\n")

--- Trial Store Performance vs. Scaled Control ---
Trial Store: 77 | Control Store: 187
  Scaled Control Sales (Expected): $738.0
  Trial Store Actual Sales:        $777.0
  Sales Difference:                +$39.0
  Performance Improvement:         +5.29%

Trial Store: 86 | Control Store: 196
  Scaled Control Sales (Expected): $2617.92
  Trial Store Actual Sales:        $2788.2
  Sales Difference:                +$170.28
  Performance Improvement:         +6.5%

Trial Store: 88 | Control Store: 237
  Scaled Control Sales (Expected): $3832.94
  Trial Store Actual Sales:        $4286.8
  Sales Difference:                +$453.86
  Performance Improvement:         +11.84%

